# Deploy Agent to Bedrock AgentCore Runtime

This notebook deploys the MCP agent from the [previous notebook](bedrock_agentcore_mcp_agent.ipynb) as a **managed service** on Amazon Bedrock AgentCore Runtime — entirely from within this notebook, no terminal required.

You'll learn how to:
- Package an agent for AgentCore Runtime using `direct_code_deploy`
- Use `%%writefile` magic to create deployment files from notebook cells
- Deploy and invoke the agent using `!` shell commands
- Invoke the deployed agent programmatically via boto3

**Architecture:**
```
User → AgentCore Runtime → ReAct Agent (Claude) → MCP Gateway → Neo4j Database
```

**Prerequisites:**
- `../CONFIG.txt` configured with `MODEL_ID`, `REGION`, `MCP_GATEWAY_URL`, and `MCP_ACCESS_TOKEN`
- AWS credentials with Bedrock and AgentCore access

## 1. Load Configuration

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

# Validate required config
missing = []
if not MODEL_ID: missing.append("MODEL_ID")
if not MCP_GATEWAY_URL: missing.append("MCP_GATEWAY_URL")
if not MCP_ACCESS_TOKEN: missing.append("MCP_ACCESS_TOKEN")
if missing:
    raise ValueError(f"Missing required config in ../CONFIG.txt: {', '.join(missing)}")

print(f"Model:   {MODEL_ID}")
print(f"Region:  {REGION}")
print(f"Gateway: {MCP_GATEWAY_URL[:60]}...")

## 2. Install AgentCore Toolkit

The `bedrock-agentcore-starter-toolkit` provides the `agentcore` CLI for configuring, deploying, and invoking agents on AgentCore Runtime.

In [ ]:
%pip install bedrock-agentcore-starter-toolkit>=0.3.3 bedrock-agentcore>=1.4.7 langchain>=1.2.0 langgraph>=1.0.6 langchain-aws>=1.2.0 langchain-mcp-adapters>=0.2.0 mcp>=1.3.0 pyyaml -q

## 3. Create Agent Directory

AgentCore's `direct_code_deploy` packages an entire directory and uploads it. We'll create a self-contained directory with:
- `agent.py` — the agent code with `BedrockAgentCoreApp` handler
- `pyproject.toml` — Python dependencies
- `.mcp-credentials.json` — MCP Gateway credentials

In [ ]:
!mkdir -p agentcore_deploy
print("Created agentcore_deploy/ directory")

## 4. Write the Agent Code

The agent uses the same architecture as the [MCP agent notebook](bedrock_agentcore_mcp_agent.ipynb), wrapped in a `BedrockAgentCoreApp` handler that AgentCore Runtime expects:

1. Load MCP credentials and connect to the Neo4j MCP server via AgentCore Gateway
2. Build a ReAct agent with `create_react_agent` from LangGraph
3. Process incoming prompts and yield chunked responses

The `@app.entrypoint` decorator registers the `invoke` function as the handler for incoming requests.

In [ ]:
%%writefile agentcore_deploy/agent.py
#!/usr/bin/env python3
"""
Finance Agent - AgentCore Runtime Deployment

A ReAct agent for SEC financial data analysis that connects to the Neo4j MCP
server via AgentCore Gateway.
"""

import json
import logging
import os
from pathlib import Path

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

app = BedrockAgentCoreApp()

CREDENTIALS_FILE = ".mcp-credentials.json"
MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-4-20250514-v1:0")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

SYSTEM_PROMPT = """You are a financial analysis assistant with access to a Neo4j knowledge graph \
containing SEC filing data, company information, risk factors, and institutional ownership.

When answering questions about the database:
1. First retrieve the schema to understand the database structure
2. Formulate appropriate Cypher queries based on the actual schema
3. Format results in a clear, human-readable way
4. Cite specific data from the graph

Always use LIMIT on non-aggregation queries (10-25 for browsing, 5 for samples).
Never execute write queries."""


def load_credentials() -> dict:
    """Load credentials from .mcp-credentials.json."""
    path = Path(CREDENTIALS_FILE)
    if not path.exists():
        raise FileNotFoundError(
            "Credentials file not found: .mcp-credentials.json"
        )
    with open(path) as f:
        return json.load(f)


def get_llm(region: str = "us-east-1"):
    return init_chat_model(
        MODEL_ID,
        model_provider="bedrock_converse",
        region_name=region,
        temperature=0,
    )


@app.entrypoint
async def invoke(payload: dict = None):
    """AgentCore Runtime handler - processes financial queries via Neo4j MCP."""
    if payload is None:
        payload = {}

    prompt = (
        payload.get("prompt")
        or payload.get("message")
        or payload.get("query")
        or payload.get("input")
    )

    if not prompt:
        yield {"type": "error", "error": "No prompt provided. Include 'prompt' in your request."}
        return

    logger.info(f"Query: {prompt[:100]}...")

    try:
        credentials = load_credentials()
        gateway_url = credentials["gateway_url"]
        access_token = credentials["access_token"]
        region = credentials.get("region", AWS_REGION)

        logger.info(f"Gateway: {gateway_url}")

        llm = get_llm(region)

        mcp_client = MultiServerMCPClient(
            {
                "neo4j": {
                    "transport": "streamable_http",
                    "url": gateway_url,
                    "headers": {
                        "Authorization": f"Bearer {access_token}",
                    },
                }
            }
        )

        tools = await mcp_client.get_tools()
        logger.info(f"Loaded {len(tools)} tools: {[t.name for t in tools]}")

        agent = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)

        result = await agent.ainvoke({"messages": [("human", prompt)]})

        messages = result.get("messages", [])
        if messages and hasattr(messages[-1], "content"):
            response_text = messages[-1].content
        else:
            response_text = "No response from agent"

        yield {"type": "chunk", "data": response_text}
        yield {"type": "complete"}

        logger.info("Request completed successfully")

    except FileNotFoundError as e:
        logger.error(f"Credentials error: {e}")
        yield {"type": "error", "error": str(e)}
    except Exception as e:
        logger.error(f"Error: {e}", exc_info=True)
        yield {"type": "error", "error": f"Error processing request: {str(e)}"}


if __name__ == "__main__":
    app.run(port=8080)

## 5. Write Dependencies

In [ ]:
%%writefile agentcore_deploy/pyproject.toml
[project]
name = "finance-agent"
version = "0.1.0"
description = "Finance ReAct agent for SEC data via Neo4j MCP and AgentCore Gateway"
requires-python = ">=3.10"
dependencies = [
    "langchain>=1.2.0",
    "langgraph>=1.0.6",
    "langchain-aws>=1.2.0",
    "langchain-mcp-adapters>=0.2.0",
    "mcp>=1.3.0",
    "boto3>=1.42.0",
    "bedrock-agentcore>=1.4.7",
]

## 6. Write MCP Credentials

The agent reads MCP Gateway credentials from `.mcp-credentials.json`. We'll create this file from the values in `CONFIG.txt` so they're packaged with the agent code on deployment.

In [ ]:
import json

credentials = {
    "gateway_url": MCP_GATEWAY_URL,
    "access_token": MCP_ACCESS_TOKEN,
    "region": REGION,
}

creds_path = "agentcore_deploy/.mcp-credentials.json"
with open(creds_path, "w") as f:
    json.dump(credentials, f, indent=2)

print(f"Wrote {creds_path}")
print(f"  gateway_url: {MCP_GATEWAY_URL[:60]}...")
print(f"  region:      {REGION}")

Verify all deployment files are in place:

In [ ]:
!ls -la agentcore_deploy/

## 7. Configure for Deployment

AgentCore needs a `.bedrock_agentcore.yaml` config file that specifies the entrypoint, deployment type, and AWS settings. We generate this programmatically to avoid interactive CLI prompts that don't work in notebooks.

In [ ]:
import boto3
import yaml

# Get AWS account ID
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

# Build absolute paths for AgentCore config
agent_dir = os.path.abspath("agentcore_deploy")
entrypoint = os.path.join(agent_dir, "agent.py")

AGENT_NAME = "finance_agent"

config = {
    "default_agent": AGENT_NAME,
    "agents": {
        AGENT_NAME: {
            "name": AGENT_NAME,
            "language": "python",
            "entrypoint": entrypoint,
            "deployment_type": "direct_code_deploy",
            "runtime_type": "PYTHON_3_13",
            "platform": "linux/arm64",
            "source_path": agent_dir,
            "aws": {
                "account": account_id,
                "region": REGION,
                "execution_role_auto_create": True,
                "ecr_auto_create": False,
                "s3_auto_create": True,
                "network_configuration": {
                    "network_mode": "PUBLIC",
                },
                "protocol_configuration": {
                    "server_protocol": "HTTP",
                },
                "observability": {
                    "enabled": True,
                },
            },
        }
    },
}

config_path = os.path.join(agent_dir, ".bedrock_agentcore.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Wrote {config_path}")
print(f"  Agent:      {AGENT_NAME}")
print(f"  Account:    {account_id}")
print(f"  Region:     {REGION}")
print(f"  Deploy:     direct_code_deploy")
print(f"  IAM Role:   auto-create")
print(f"  S3 Bucket:  auto-create")

## 8. Deploy to AgentCore Runtime

This packages the agent code and deploys it to AgentCore Runtime as a managed service. The deployment uses `direct_code_deploy` (no Docker required) and configures CloudWatch logging automatically.

In [ ]:
!cd agentcore_deploy && agentcore deploy

## 9. Invoke the Deployed Agent

Once deployed, you can invoke the agent via the `agentcore` CLI or programmatically with boto3.

### Option A: Invoke via CLI

In [ ]:
!cd agentcore_deploy && agentcore invoke '{"prompt": "What companies are in the database?"}'

In [ ]:
!cd agentcore_deploy && agentcore invoke '{"prompt": "What are Apple\'s key risk factors?"}'

In [ ]:
!cd agentcore_deploy && agentcore invoke '{"prompt": "Who are the largest institutional owners of NVIDIA?"}'

### Option B: Invoke via boto3

For programmatic access, use the `bedrock-agentcore` boto3 client. First, extract the agent ARN from the deployment config:

In [ ]:
import yaml

with open("agentcore_deploy/.bedrock_agentcore.yaml") as f:
    config = yaml.safe_load(f)

# Get the default agent's ARN
default_agent = config["default_agent"]
agent_config = config["agents"][default_agent]
AGENT_ARN = agent_config["bedrock_agentcore"]["agent_arn"]
AGENT_REGION = agent_config["aws"]["region"]

print(f"Agent:  {default_agent}")
print(f"ARN:    {AGENT_ARN}")
print(f"Region: {AGENT_REGION}")

In [ ]:
import json
import uuid
import boto3


def invoke_agent(prompt: str):
    """Invoke the deployed agent via boto3."""
    client = boto3.client("bedrock-agentcore", region_name=AGENT_REGION)

    response = client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
        qualifier="DEFAULT",
    )

    content = [chunk.decode("utf-8") for chunk in response.get("response", [])]
    result = json.loads("".join(content))
    return result


print("invoke_agent() function ready")

In [ ]:
result = invoke_agent("What companies are in the database and what are their key risk factors?")
print(result)

## 10. Cleanup

When you're done, remove the agent from AgentCore Runtime. This deletes the deployed runtime but keeps your local files.

In [ ]:
# Uncomment the line below to destroy the deployed agent
# !cd agentcore_deploy && agentcore destroy